In [ ]:
import os
import shutil
import logging
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

In [ ]:
logging.basicConfig(
    filename="robo_cadastro.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding="utf-8")

logging.info("Iniciando o robô de cadastro.")

In [ ]:
arquivo_excel = os.path.abspath("alunos_projeto_final.xlsx")
arquivo_html = os.path.abspath("index.html")

print("Arquivo Excel:")
print(arquivo_excel)

print("\nArquivo HTML:")
print(arquivo_html)

In [ ]:
if not os.path.exists(arquivo_excel):
    raise FileNotFoundError("O arquivo alunos_projeto_final.xlsx não foi encontrado.")
if not os.path.exists(arquivo_html):
    raise FileNotFoundError("O arquivo index.html não foi encontrado.")

print("Arquivos encontrados com sucesso.")

In [ ]:
df = pd.read_excel(arquivo_excel)
print("Dados carregados com sucesso.")
print(f"Quantidade de alunos: {len(df)}")
df

In [ ]:
print("Colunas encontradas:")
for coluna in df.columns:
    print("-", coluna)

In [ ]:
nao_cadastrados = []
sucessos = []

print("Listas de controle preparadas.")

In [ ]:
driver = webdriver.Chrome()
driver.maximize_window()
print("Chrome iniciado com sucesso.")

In [ ]:
driver.get(f"file:///{arquivo_html.replace(os.sep, '/')}")
print("Sistema web aberto.")
print("Título da página:", driver.title)

In [ ]:
for _, aluno in df.iterrows():

    nome = aluno["Nome"]
    cpf = aluno["CPF"]
    whatsapp = aluno["WhatsApp"]

    print(f"Processando: {nome}")

    # Verifica se o WhatsApp está preenchido
    if pd.isna(whatsapp) or str(whatsapp).strip() == "":
        
        mensagem = "WhatsApp não informado"

        print(f"  Pendente: {mensagem}")

        logging.warning(f"Aluno não cadastrado: {nome} - CPF: {cpf} - Motivo: {mensagem}")

        nao_cadastrados.append({"Nome": nome,"CPF": cpf,"Status": mensagem})
        continue

    try:
        campo_nome = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "nome")))
        campo_whatsapp = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "whatsapp")))
        campo_nome.clear()
        campo_whatsapp.clear()

        campo_nome.send_keys(str(nome))
        campo_whatsapp.send_keys(str(whatsapp))

        botao_cadastrar = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "cadastrar")))
        botao_cadastrar.click()

        logging.info(f"Aluno cadastrado com sucesso: {nome} - CPF: {cpf}")

        sucessos.append({"Nome": nome,"CPF": cpf,"WhatsApp": whatsapp,"Status": "Cadastrado com sucesso"})

        print("Cadastro realizado.")

    except Exception as erro:
        logging.error(f"Falha ao cadastrar o aluno: {nome} - CPF: {cpf} - Erro: {erro}")

        nao_cadastrados.append({"Nome": nome,"CPF": cpf,"Status": f"Erro no cadastro: {erro}"})

        print(f"Erro: {erro}")

print("\nProcessamento concluído.")

In [ ]:
df_pendencias = pd.DataFrame(nao_cadastrados)
print(f"Total de pendências: {len(df_pendencias)}")
df_pendencias

In [ ]:
html_atualizado = driver.page_source
print("Código-fonte atualizado capturado.")

In [ ]:
soup = BeautifulSoup(html_atualizado, "html.parser")
print("HTML processado com BeautifulSoup.")

In [ ]:
corpo_tabela = soup.find("tbody", id="corpo_tabela")
if corpo_tabela:
    print("Tabela de histórico encontrada.")
else:
    print("Tabela de histórico não encontrada.")

In [ ]:
sucessos_historico = []

if corpo_tabela:

    linhas = corpo_tabela.find_all("tr")

    for linha in linhas:
        colunas = linha.find_all("td")
        if len(colunas) >= 3:
            registro = {
                "Nome": colunas[0].get_text(strip=True),
                "CPF": colunas[1].get_text(strip=True),
                "WhatsApp": colunas[2].get_text(strip=True)}
            sucessos_historico.append(registro)

print(f"Registros encontrados no histórico: {len(sucessos_historico)}")

In [ ]:
df_sucesso = pd.DataFrame(sucessos_historico)
df_sucesso

In [ ]:
driver.quit()
print("Navegador encerrado.")
logging.info("Navegador encerrado após o processamento.")

In [ ]:
arquivo_pendencias = "relatorio_pendencias.xlsx"
df_pendencias.to_excel(arquivo_pendencias,index=False)

print(f"Relatório de pendências salvo em: {arquivo_pendencias}")
logging.info(f"Relatório de pendências gerado: {arquivo_pendencias}")

In [ ]:
arquivo_sucesso = "relatorio_sucesso_historico.xlsx"
df_sucesso.to_excel(arquivo_sucesso,index=False)

print(f"Relatório de sucesso salvo em: {arquivo_sucesso}")
logging.info(f"Relatório de sucesso gerado: {arquivo_sucesso}")

In [ ]:
pasta_processados = "processados"

os.makedirs(pasta_processados,exist_ok=True)
print(f"Pasta preparada: {pasta_processados}")

In [ ]:
shutil.copy(arquivo_pendencias,os.path.join(pasta_processados, arquivo_pendencias))
logging.info(f"Relatório movido para processados: {arquivo_pendencias}")

shutil.copy(arquivo_sucesso,os.path.join(pasta_processados, arquivo_sucesso))
logging.info(f"Relatório movido para processados: {arquivo_sucesso}")

print("Relatórios copiados para a pasta processados.")

In [ ]:
total = len(df)
sucesso = len(df_sucesso)
pendentes = len(df_pendencias)

print("=======================")
print("RESUMO DO PROCESSAMENTO")
print("=======================")

print(f"Total de alunos: {total}")
print(f"Cadastros realizados: {sucesso}")
print(f"Pendências: {pendentes}")

if total > 0:
    taxa_sucesso = (sucesso / total) * 100
    print(f"Taxa de sucesso: {taxa_sucesso:.2f}%")

print("=======================")

logging.info(
    f"Processamento finalizado - Total: {total} | "
    f"Sucessos: {sucesso} | Pendências: {pendentes}"
)

In [ ]:
print("DIÁRIO DO ROBÔ")
print("==============")

try:
    with open("robo_cadastro.log", "r", encoding="utf-8") as arquivo_log:
        print(arquivo_log.read())
except Exception as erro:
    print(f"Não foi possível abrir o arquivo de log: {erro}")